### Instalação das dependências

O H2O AutoML requer Java 8 ou superior instalado. No Google Colab, o Java já está disponível por padrão. Execute esta célula antes de qualquer outra.

In [ ]:
!pip install h2o pandas -q

# Código 2.3 — Automação de ML com H2O AutoML

**Objetivo:** Demonstrar o treinamento automatizado de múltiplos modelos e geração de leaderboard utilizando o framework **H2O AutoML**.

### Passo 1: Inicialização do Cluster H2O

Inicializamos o motor H2O em memória.

In [ ]:
import h2o
from h2o.automl import H2OAutoML

# Inicializar o cluster H2O local
h2o.init(max_mem_size="2G")

**Entendendo os parâmetros de inicialização do H2O:**
- `h2o.init(max_mem_size="2G")`: Inicializa o servidor Java do H2O reservando até 2 Gigabytes de memória RAM.

### Passo 2: Carga e Conversão dos Dados em `H2OFrame`

Convertemos o dataset Pandas em um quadro de dados nativo do H2O.

In [ ]:
# Carregar dados usando Pandas e converter para H2OFrame
import pandas as pd
df_pandas = pd.read_csv("dados_churn.csv")
hf = h2o.H2OFrame(df_pandas)

# Definir a coluna alvo como fator (classificação)
hf["churn"] = hf["churn"].asfactor()

# Obter lista de colunas preditoras e alvo
predictors = [col for col in hf.columns if col != "churn"]
target = "churn"

# Divisão Treino / Teste no H2O
train, test = hf.split_frame(ratios=[0.8], seed=42)
print(f"Linhas em Treino: {train.nrows}, Linhas em Teste: {test.nrows}")

**Entendendo a preparação de dados no H2O:**
- `asfactor()`: Converte a coluna numérica em tipo categórico (*factor*), instruindo o H2O a tratar a tarefa como classificação e não regressão.
- `split_frame(ratios=[0.8])`: Separa 80% dos dados para o conjunto de treinamento no motor H2O.

### Passo 3: Configuração e Execução do H2O AutoML

Executamos a busca automatizada limitada a 60 segundos ou no máximo 10 modelos.

In [ ]:
# Configurar a busca AutoML
aml = H2OAutoML(
    max_models=10,
    max_runtime_secs=60,
    seed=42,
    sort_metric="AUC"
)

# Treinar os modelos
aml.train(x=predictors, y=target, training_frame=train)

# Exibir a Tabela de Líderes (Leaderboard)
lb = aml.leaderboard
print("=== H2O AUTOML LEADERBOARD ===")
print(lb.head(rows=10))

**Entendendo os parâmetros do `H2OAutoML`:**
- `max_models=10`: Limita a busca ao treinamento de no máximo 10 algoritmos/modelos diferentes.
- `max_runtime_secs=60`: Interrompe o processo de busca após 60 segundos de execução.
- `sort_metric="AUC"`: Ordena a tabela final de classificação (*Leaderboard*) pela métrica AUC.

### Passo 4: Avaliação do Melhor Modelo (*Leader*)

Avaliamos o melhor modelo encontrado no conjunto de teste.

In [ ]:
# Obter o melhor modelo
best_model = aml.leader

# Performance no teste
perf = best_model.model_performance(test)
print(f"AUC do Melhor Modelo no Teste: {perf.auc():.4f}")

# Finalizar cluster H2O
h2o.cluster().shutdown()

---

## Conclusão da Aula — H2O AutoML

Ferramentas de AutoML como o **H2O** aceleram a fase de exploração e estabelecem uma linha de base (*baseline*) competitiva rapidamente, testando dezenas de combinações de algoritmos de forma automatizada.